# Masked self-attention
1. Self-attention on the decoder input "<start> comment ça va ?" (the French of "How are you?"):
   without a mask every word takes weight from the words after it.
2. The causal mask: add -inf above the diagonal before the softmax. Future weights become exactly 0.
3. Checked against keras.layers.MultiHeadAttention(use_causal_mask=True).
4. No leakage: changing the future words leaves the earlier outputs unchanged.
5. One masked pass = n separate passes over the growing prefix (what inference does), and the time each takes.

In [1]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
import time
from pathlib import Path
import numpy as np
import pandas as pd
import keras
import tensorflow as tf

tf.config.experimental.enable_tensor_float_32_execution(False)   # full float32 precision on a GPU
np.set_printoptions(precision=3, suppress=True)
Path("data").mkdir(exist_ok=True)


def softmax_rows(S):
    W = np.exp(S - S.max(-1, keepdims=True))      # exp(-inf) = 0
    return W / W.sum(-1, keepdims=True)

I0000 00:00:1791081541.089872 2054490 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


## 1. The sentence pair and the decoder input
A real pair of the Keras English-French corpus (Tatoeba). With teacher forcing the decoder input is the
French sentence shifted right by one place, starting with <start>.

In [2]:
path = keras.utils.get_file("fra-eng.zip", origin="http://storage.googleapis.com/download.tensorflow.org/data/fra-eng.zip",
                            extract=True)
lines = Path(path, "fra.txt").read_text(encoding="utf-8").split("\n")
pair = next(l.split("\t")[:2] for l in lines if l.startswith("How are you?\tComment ça va ?"))
print(pair)
words = pair[1].lower().split()
dec_in = ["<start>"] + words
dec_out = words + ["<end>"]
print(pd.DataFrame([dec_in, dec_out], index=["decoder input", "target"]))

['How are you?', 'Comment ça va ?']
                     0        1   2   3      4
decoder input  <start>  comment  ça  va      ?
target         comment       ça  va   ?  <end>


## 2. Self-attention without a mask
Each word gets 8 random numbers as a stand-in for its embedding plus positional encoding.
W_Q, W_K, W_V are the (random, untrained) weights of a one-head Keras layer; its output matrix is set to the identity.

In [3]:
keras.utils.set_random_seed(0)                # same random W_Q, W_K, W_V on every run
rng = np.random.default_rng(0)
d, n = 8, len(dec_in)
X = rng.normal(size=(n, d)).astype("float32")
mha = keras.layers.MultiHeadAttention(num_heads=1, key_dim=d, use_bias=False)
mha(X[None], X[None])
Wq, Wk, Wv, _ = mha.get_weights()
mha.set_weights([Wq, Wk, Wv, np.eye(d, dtype="float32")[None]])
Wq, Wk, Wv = Wq[:, 0], Wk[:, 0], Wv[:, 0]
Q, K, V = X @ Wq, X @ Wk, X @ Wv
S = Q @ K.T / np.sqrt(d)
A_full = softmax_rows(S)
print("scaled scores:\n", S)
print("weights without a mask (rows: the word being computed, columns: the words it takes from):\n", A_full)
print("weight that each row takes from LATER words:", np.triu(A_full, 1).sum(1))

scaled scores:
 [[-0.239  0.164 -0.303  0.801 -0.075]
 [-0.663 -0.306  0.534 -1.321 -0.123]
 [-0.066  0.312 -0.048  0.264  0.2  ]
 [ 0.344 -0.351 -0.145 -0.25   0.245]
 [ 0.014  0.507 -0.146  0.198  0.295]]
weights without a mask (rows: the word being computed, columns: the words it takes from):
 [[0.134 0.201 0.126 0.38  0.158]
 [0.125 0.179 0.415 0.065 0.215]
 [0.162 0.236 0.165 0.225 0.211]
 [0.28  0.14  0.172 0.155 0.254]
 [0.166 0.272 0.142 0.2   0.22 ]]
weight that each row takes from LATER words: [0.866 0.695 0.437 0.254 0.   ]


## 3. The mask
M has 0 on and below the diagonal and -inf above it. Add it to the scores, then take the softmax.

In [4]:
M = np.triu(np.full((n, n), -np.inf), 1)
print("mask M:\n", M)
A_mask = softmax_rows(S + M)
print("masked weights:\n", A_mask)
print("row sums:", A_mask.sum(1))
assert (A_mask[np.triu_indices(n, 1)] == 0).all()          # every future weight is exactly 0
Y_mask = A_mask @ V
pd.DataFrame(A_full, index=dec_in, columns=dec_in).round(4).to_csv("data/weights_unmasked.csv")
pd.DataFrame(A_mask, index=dec_in, columns=dec_in).round(4).to_csv("data/weights_masked.csv")

mask M:
 [[  0. -inf -inf -inf -inf]
 [  0.   0. -inf -inf -inf]
 [  0.   0.   0. -inf -inf]
 [  0.   0.   0.   0. -inf]
 [  0.   0.   0.   0.   0.]]
masked weights:
 [[1.    0.    0.    0.    0.   ]
 [0.412 0.588 0.    0.    0.   ]
 [0.288 0.42  0.293 0.    0.   ]
 [0.375 0.187 0.23  0.207 0.   ]
 [0.166 0.272 0.142 0.2   0.22 ]]
row sums: [1. 1. 1. 1. 1.]


Worked example, row of "ça" (position 3): the scores, the masked scores, the weights.

In [5]:
r = 2
print("scores:       ", S[r])
print("scores + mask:", S[r] + M[r])
e = np.exp(S[r] + M[r])
print("exp:          ", e, " sum", e.sum().round(3))
print("weights:      ", A_mask[r])

scores:        [-0.066  0.312 -0.048  0.264  0.2  ]
scores + mask: [-0.066  0.312 -0.048   -inf   -inf]
exp:           [0.936 1.366 0.953 0.    0.   ]  sum 3.255
weights:       [0.288 0.42  0.293 0.    0.   ]


## 4. The same with Keras: use_causal_mask=True
Keras replaces masked scores by -1e9 before the softmax and then sets the masked weights to 0.

In [6]:
out_k, w_k = mha(X[None], X[None], use_causal_mask=True, return_attention_scores=True)
w_k = w_k.numpy()[0, 0]
print("largest difference, weights:", np.abs(w_k - A_mask).max(), " outputs:", np.abs(out_k.numpy()[0] - Y_mask).max())
assert np.allclose(w_k, A_mask, atol=1e-6) and np.allclose(out_k.numpy()[0], Y_mask, atol=1e-5)
assert (w_k[np.triu_indices(n, 1)] == 0).all()

largest difference, weights: 3.251710908869221e-08  outputs: 8.420771135941862e-08


## 5. No leakage: change the future, the past stays the same
Replace the last two input words ("va", "?") by two other words (new random vectors).

In [7]:
X2 = X.copy()
X2[3:] = rng.normal(size=(n - 3, d))
for name, use_mask in (("without mask", False), ("with mask", True)):
    y1 = mha(X[None], X[None], use_causal_mask=use_mask).numpy()[0]
    y2 = mha(X2[None], X2[None], use_causal_mask=use_mask).numpy()[0]
    print(f"{name}: change of the outputs at positions 1-3 (<start>, comment, ça):", np.abs(y1 - y2)[:3].max(1).round(4))
y1 = mha(X[None], X[None], use_causal_mask=True).numpy()[0]
y2 = mha(X2[None], X2[None], use_causal_mask=True).numpy()[0]
assert np.array_equal(y1[:3], y2[:3])

without mask: change of the outputs at positions 1-3 (<start>, comment, ça): [0.479 1.699 0.521]


with mask: change of the outputs at positions 1-3 (<start>, comment, ça): [0. 0. 0.]


## 6. One masked pass = n passes over the growing prefix
At inference the decoder only has the words written so far. Run unmasked attention on the prefix
dec_in[:t] and keep its last row. The n results equal the n rows of the single masked pass.

In [8]:
def attend(Xp, mask):
    Qp, Kp, Vp = Xp @ Wq, Xp @ Wk, Xp @ Wv
    Sp = Qp @ Kp.T / np.sqrt(Xp.shape[1])
    if mask:
        Sp = Sp + np.triu(np.full(Sp.shape, -np.inf), 1)
    return softmax_rows(Sp) @ Vp


Y_seq = np.stack([attend(X[:t], mask=False)[-1] for t in range(1, n + 1)])
print("largest difference, prefix by prefix vs one masked pass:", np.abs(Y_seq - attend(X, mask=True)).max())
assert np.allclose(Y_seq, attend(X, mask=True), atol=1e-6)

largest difference, prefix by prefix vs one masked pass: 1.00657215451605e-07


Timing on the GPU, as in training: a batch of 64 sequences, d = 64, one head (TensorFlow).
Parallel: one masked pass over all n positions. Step by step: n steps, one new word per step.
To keep the comparison fair the step-by-step version does no repeated work: keys and values of all words
are computed once, and step t only computes the query of word t and its row of weights.
Each time is the fastest of 5 repeats, after one warm-up run.

In [9]:
print(tf.config.list_physical_devices("GPU"))
g = tf.random.Generator.from_seed(1)
Wq_t, Wk_t, Wv_t = (g.normal((64, 64)) / 8 for _ in range(3))


from functools import lru_cache


@lru_cache
def mask_matrix(L):
    upper = np.triu(np.full((L, L), -np.inf, dtype="float32"), 1)
    return tf.constant(upper)


def parallel(XL):
    S = tf.einsum("bid,bjd->bij", XL @ Wq_t, XL @ Wk_t) / 8.0 + mask_matrix(XL.shape[1])
    return tf.nn.softmax(S, axis=-1) @ (XL @ Wv_t)


def step_by_step(XL):
    Kall, Vall = XL @ Wk_t, XL @ Wv_t
    out = []
    for t in range(1, XL.shape[1] + 1):
        q = XL[:, t - 1:t] @ Wq_t                                    # (batch, 1, d)
        w = tf.nn.softmax(tf.einsum("bid,bjd->bij", q, Kall[:, :t]) / 8.0, axis=-1)
        out.append(w @ Vall[:, :t])
    return tf.concat(out, axis=1)


def best(f, reps=5):
    f().numpy()
    t = []
    for _ in range(reps):
        t0 = time.perf_counter(); f().numpy(); t.append(time.perf_counter() - t0)
    return min(t)


rows = []
for L in (8, 32, 128, 512):
    XL = g.normal((64, L, 64))
    assert np.allclose(step_by_step(XL).numpy(), parallel(XL).numpy(), atol=1e-4)
    par = best(lambda: parallel(XL))
    seq = best(lambda: step_by_step(XL))
    rows.append(dict(length=L, parallel_ms=round(par * 1e3, 2), step_by_step_ms=round(seq * 1e3, 2), ratio=round(seq / par, 1)))
timing = pd.DataFrame(rows)
timing.to_csv("data/timing.csv", index=False)
print(timing)

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


   length  parallel_ms  step_by_step_ms  ratio
0       8         5.09             5.87    1.2
1      32         7.04            18.03    2.6
2     128         5.70            68.13   11.9
3     512        16.54           283.90   17.2
